# ManyFEWS — flood forecast

**Runtime → Run all.** About 90 seconds, most of it the initial download.

Produces a river flow forecast and, if flooding is predicted, a flood depth map
for Majalaya, West Java. Live data from Open-Meteo; no accounts or API keys.

Edit the parameters in cell 3 to change location, horizon or scenario.

In [ ]:
# Put the core package on the path, cloning the repository (~25 MB of model
# data) if we are not already inside a checkout.
import subprocess, sys
from pathlib import Path

REPO = "https://github.com/simreaney/ManyFEWS.git"
BRANCH = "main"          # set this if core/ lives on a different branch

def find_core():
    """Look for core/manyfews_core here, in any parent, or in a clone."""
    for base in (Path.cwd(), *Path.cwd().parents, Path("ManyFEWS")):
        candidate = base / "core"
        if (candidate / "manyfews_core" / "__init__.py").is_file():
            return candidate
    return None

core = find_core()
if core is None:
    if not Path("ManyFEWS").is_dir():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO], check=True
        )
    core = find_core()

if core is None:
    raise SystemExit(
        f"Could not find the manyfews_core package.\n\n"
        f"The clone of {REPO} (branch {BRANCH}) has no core/ directory, so there "
        f"is nothing to import.\n"
        f"That directory has to be committed and pushed before this notebook can "
        f"run in Colab.\n"
        f"If it lives on another branch, set BRANCH above and re-run this cell."
    )

sys.path.insert(0, str(core))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "folium"], check=True)

import manyfews_core as mf
from manyfews_core.plotting import apply_style
apply_style()

print(f"manyfews_core {mf.__version__}")
print(f"data directory: {mf.data_dir()}")
for name in ("RainfallRunoffModelParameters.csv",
             "floodEmulatorParams-20230921.csv",
             "channel.geojson"):
    size = mf.data_path(name).stat().st_size / 1e6
    print(f"  {name:<38} {size:7.2f} MB")

## Parameters

In [ ]:
# --- catchment -------------------------------------------------------------
WEATHER_LAT      = -7.05        # where the forecast is sampled
WEATHER_LON      = 107.758
CATCHMENT_LAT    = -7.125       # catchment mean latitude
CATCHMENT_ALT_M  = 1157.0       # catchment mean altitude
CATCHMENT_AREA   = 212.2640     # km²

# --- forecast --------------------------------------------------------------
FORECAST_DAYS    = 16
ENSEMBLE_MEMBERS = 10           # 0 keeps every member Open-Meteo offers
SPINUP_DAYS      = 29

# --- scenario --------------------------------------------------------------
STORM_ENABLED    = False        # True injects a synthetic design storm
STORM_TOTAL_MM   = 200.0        # note: 100 mm is not enough to flood here
STORM_DAYS_AHEAD = 2

# --- output ----------------------------------------------------------------
MAP_PERCENTILE   = 90.0         # 90 = cautious, 50 = central estimate
MAP_MAX_DEPTH_M  = 3.0
MASK_CHANNEL     = True

CATCHMENT = mf.CatchmentConfig(
    latitude_deg=CATCHMENT_LAT, altitude_m=CATCHMENT_ALT_M,
    area_km2=CATCHMENT_AREA, weather_lat=WEATHER_LAT, weather_lon=WEATHER_LON,
)
FORECAST = mf.ForecastConfig(
    forecast_days=FORECAST_DAYS, max_members=ENSEMBLE_MEMBERS, spinup_days=SPINUP_DAYS,
)
STORM = mf.StormConfig(
    enabled=STORM_ENABLED, total_mm=STORM_TOTAL_MM, days_ahead=STORM_DAYS_AHEAD,
)

print(f"catchment {CATCHMENT.area_km2:.0f} km² at {WEATHER_LAT}, {WEATHER_LON}")
print(f"{FORECAST_DAYS}-day forecast, {ENSEMBLE_MEMBERS} members")
print("synthetic storm: " + (f"{STORM_TOTAL_MM:.0f} mm on day {STORM_DAYS_AHEAD}"
                             if STORM_ENABLED else "off"))

## Run the forecast

In [ ]:
import time
import numpy as np

clock = time.time()

params = mf.load_parameters()
print("· loading model parameters")

history = mf.fetch_history(CATCHMENT, FORECAST)
print(f"· fetched {len(history) // 4} days of observed weather   [{time.time() - clock:5.1f}s]")

state = mf.spin_up(history, params, CATCHMENT)
print(f"· spun up catchment state                    [{time.time() - clock:5.1f}s]")

forecast = mf.fetch_forecast(CATCHMENT, FORECAST)
print(f"· fetched {len(forecast)} ensemble members            [{time.time() - clock:5.1f}s]")

if STORM.enabled:
    forecast = mf.inject_storm_ensemble(forecast, forecast[0].start, STORM)
    print(f"· injected {STORM.total_mm:.0f} mm synthetic storm")

ensemble = mf.run_ensemble(forecast, state, params, CATCHMENT)
print(f"· ran {ensemble.flow_m3s.shape[0]}×{ensemble.flow_m3s.shape[2]} model realisations  "
      f"[{time.time() - clock:5.1f}s]")

emulator = mf.FloodEmulator.from_csv()
channel = mf.cached_channel_mask(emulator) if MASK_CHANNEL else None
print(f"· loaded {emulator.n_cells:,}-cell flood model       [{time.time() - clock:5.1f}s]")

## River flow forecast

In [ ]:
import matplotlib.pyplot as plt
from manyfews_core.plotting import forecast_panel

forecast_panel(ensemble, storm=STORM)
plt.show()

## Daily summary

In [ ]:
from manyfews_core.plotting import FLOOD_THRESHOLD_M3S

rows = []
for day in range(len(ensemble.times) // 4):
    steps = range(day * 4, day * 4 + 4)
    pooled = np.concatenate([ensemble.pooled(s) for s in steps])
    p50, p90 = np.percentile(pooled, [50, 90])
    wet = 0
    if p90 >= emulator.min_q.min():
        peak = max(steps, key=lambda s: np.percentile(ensemble.pooled(s), 90))
        wet = emulator.field(ensemble.pooled(peak),
                             channel_mask=channel).wet_cells(MAP_PERCENTILE)
    rows.append((str(ensemble.times[day * 4])[:10], p50, p90, wet,
                 mf.risk_fraction(wet)))

print(f"{'date':<12}{'flow p50':>10}{'flow p90':>10}{'flooded':>10}{'risk':>8}")
print("-" * 50)
for date, p50, p90, wet, risk in rows:
    print(f"{date:<12}{p50:>10.1f}{p90:>10.1f}{wet:>10,}{risk:>7.1%}")

worst = max(rows, key=lambda r: r[3])
peak_step = ensemble.peak_step()
peak_pooled = ensemble.pooled(peak_step)
field = emulator.field(peak_pooled, channel_mask=channel)
flooded = field.wet_cells(MAP_PERCENTILE)

def banner(title, lines):
    body = [title, ""] + lines
    width = max(len(s) for s in body) + 4
    print("\n╭" + "─" * width + "╮")
    for s in body:
        print("│  " + s.ljust(width - 2) + "│")
    print("╰" + "─" * width + "╯")

if flooded == 0:
    banner("NO FLOODING FORECAST", [
        f"Peak flow reaches {np.percentile(peak_pooled, 90):.1f} m³/s at the 90th percentile,",
        f"below the {emulator.min_q.min():.0f} m³/s at which any cell begins to flood.",
        "The map below will be empty. This is the normal result:",
        "this catchment floods rarely.",
    ])
else:
    banner("FLOODING FORECAST", [
        f"{flooded:,} cells flooded at the {MAP_PERCENTILE:.0f}th percentile, worst on {worst[0]}.",
        f"Deepest {field.max_depth(MAP_PERCENTILE):.2f} m; "
        f"mean where wet {field.mean_wet_depth(MAP_PERCENTILE):.2f} m.",
    ] + (["Driven by a synthetic storm — this is not a real forecast."]
         if STORM.enabled else []))

## Flood depth map

In [ ]:
from manyfews_core.mapping import flood_map

raster = mf.rasterise(emulator, field.layer(MAP_PERCENTILE), mask=channel)
flood_map(raster, vmax=MAP_MAX_DEPTH_M)

## Export (optional)

In [ ]:
ensemble.to_csv("river_flow_forecast.csv")
field.to_csv("flood_depths.csv")
field.to_geojson("flood_extent.geojson", pct=MAP_PERCENTILE)

import os
for name in ("river_flow_forecast.csv", "flood_depths.csv", "flood_extent.geojson"):
    print(f"{name:<28} {os.path.getsize(name) / 1e3:8.1f} kB")
print("\nDownload from the Files panel on the left.")